In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier

In [ ]:
def load_selected(path):
    df = pd.read_csv(path)
    y_class = df['Y_Class'].copy()
    drop_cols = ['Y_Quality', 'Y_Class', 'PRODUCT_ID', 'PRODUCT_CODE', 'TIMESTAMP']
    X = df.drop(columns=drop_cols)
    return X, y_class

In [ ]:
def evaluate_cv(X, y_class, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_f1 = []
    all_true, all_pred = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y_class.iloc[train_idx], y_class.iloc[val_idx]

        model = LGBMClassifier(class_weight='balanced', random_state=random_state, verbose=-1)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        all_true.extend(y_val.tolist())
        all_pred.extend(pred.tolist())

        f1 = f1_score(y_val, pred, average='macro')
        fold_f1.append(f1)
        print(f'  fold {fold} macro-F1: {f1:.4f}')

    print(f'fold 평균 macro-F1: {np.mean(fold_f1):.4f} (std {np.std(fold_f1):.4f})')
    print(f'전체 pooled macro-F1: {f1_score(all_true, all_pred, average="macro"):.4f}')
    print()
    print('confusion matrix (rows=true, cols=pred), 클래스 순서 0,1,2:')
    print(confusion_matrix(all_true, all_pred, labels=[0, 1, 2]))
    print()
    print(classification_report(all_true, all_pred, labels=[0, 1, 2], zero_division=0))

    return fold_f1, all_true, all_pred

In [ ]:
print('===== A_31 / v1 (120 features) =====')
X_A_v1, y_A_v1 = load_selected('A_31_selected.csv')
fold_f1_A_v1, true_A_v1, pred_A_v1 = evaluate_cv(X_A_v1, y_A_v1)

In [ ]:
print('===== A_31 / v2 (27 features) =====')
X_A_v2, y_A_v2 = load_selected('A_31_selected_v2.csv')
fold_f1_A_v2, true_A_v2, pred_A_v2 = evaluate_cv(X_A_v2, y_A_v2)

In [ ]:
print('===== TO_31 / v1 (52 features) =====')
X_TO_v1, y_TO_v1 = load_selected('TO_31_selected.csv')
fold_f1_TO_v1, true_TO_v1, pred_TO_v1 = evaluate_cv(X_TO_v1, y_TO_v1)

In [ ]:
print('===== TO_31 / v2 (8 features) =====')
X_TO_v2, y_TO_v2 = load_selected('TO_31_selected_v2.csv')
fold_f1_TO_v2, true_TO_v2, pred_TO_v2 = evaluate_cv(X_TO_v2, y_TO_v2)

In [ ]:
summary = pd.DataFrame({
    '조합': ['A_31 v1', 'A_31 v2', 'TO_31 v1', 'TO_31 v2'],
    'fold 평균 macro-F1': [
        np.mean(fold_f1_A_v1), np.mean(fold_f1_A_v2),
        np.mean(fold_f1_TO_v1), np.mean(fold_f1_TO_v2)
    ],
    'fold std': [
        np.std(fold_f1_A_v1), np.std(fold_f1_A_v2),
        np.std(fold_f1_TO_v1), np.std(fold_f1_TO_v2)
    ]
})
summary